In [30]:
import os
from typing import TypedDict, List, Union
from dotenv import load_dotenv

from langchain_core.messages import HumanMessage, AIMessage
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END

In [31]:
load_dotenv()

True

In [32]:

class AgentState(TypedDict):
    messages: List[Union[HumanMessage, AIMessage]]
    
llm = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0
)

In [33]:
def process(state: AgentState) -> AgentState:
    """This node will solve the request you input"""
    response = llm.invoke(state["messages"])

    state["messages"].append(AIMessage(content=response.content)) 
    print(f"\nAI: {response.content}")
    print("CURRENT STATE: ", state["messages"])

    return state

In [34]:
graph = StateGraph(AgentState)
graph.add_node("process", process)
graph.add_edge(START, "process")
graph.add_edge("process", END) 
agent = graph.compile()

In [35]:
conversation_history = []

In [36]:
user_input = input("Enter: ")
while user_input != "exit":
    conversation_history.append(HumanMessage(content=user_input))
    result = agent.invoke({"messages": conversation_history})
    conversation_history = result["messages"]
    user_input = input("Enter: ")


with open("logging.txt", "w") as file:
    file.write("Your Conversation Log:\n")
    
    for message in conversation_history:
        if isinstance(message, HumanMessage):
            file.write(f"You: {message.content}\n")
        elif isinstance(message, AIMessage):
            file.write(f"AI: {message.content}\n\n")
    file.write("End of Conversation")

print("Conversation saved to logging.txt")

Enter:  Hi, I'm Ashley



AI: <think>
Okay, the user introduced themselves as Ashley. I need to respond in a friendly and welcoming way. First, I should acknowledge their introduction. Maybe say something like "Hi Ashley!" to be warm. Then, offer assistance. Ask how I can help them today. Keep it open-ended so they can specify their needs. Make sure the tone is positive and approachable. Avoid any technical jargon. Keep sentences short and simple. Check for any typos or errors. Alright, that should cover it.
</think>

Hi Ashley! 👋 How can I assist you today? Whether you have questions, need help with something specific, or just want to chat, I'm here for you! 😊
CURRENT STATE:  [HumanMessage(content="Hi, I'm Ashley", additional_kwargs={}, response_metadata={}), AIMessage(content='<think>\nOkay, the user introduced themselves as Ashley. I need to respond in a friendly and welcoming way. First, I should acknowledge their introduction. Maybe say something like "Hi Ashley!" to be warm. Then, offer assistance. Ask h

Enter:  Good mornign



AI: <think>
Okay, the user sent "Good mornign" after I greeted them as Ashley. First, I need to correct the typo in "mornign" to "morning". Then, respond warmly. Since the user is Ashley, maybe they're starting their day and looking for a friendly interaction.

I should acknowledge their greeting, wish them a good morning in return, and offer help. Maybe add an emoji to keep it friendly. Let me check the previous message to ensure consistency. Earlier, I used a wave and a smiley, so maybe use a sun emoji here for the morning. Keep the tone positive and open-ended. Make sure the response is concise but inviting. Alright, that should cover it.
</think>

Good morning, Ashley! ☀️ Hope your day is off to a great start! How can I assist you today? 😊
CURRENT STATE:  [HumanMessage(content="Hi, I'm Ashley", additional_kwargs={}, response_metadata={}), AIMessage(content='<think>\nOkay, the user introduced themselves as Ashley. I need to respond in a friendly and welcoming way. First, I should a

Enter:  Tell me a joke



AI: <think>
Okay, Ashley asked for a joke. Let me think of a good one. I need to make sure it's appropriate and not too cheesy. Maybe a classic one that's light-hearted.

Hmm, the banana and the split? Wait, that's a play on words with "split" as in a split decision and the banana split. That's a good one. Let me check if there's a better version. Oh, maybe "Why did the banana go to the doctor? Because it wasn't peeling well!" No, that's a bit too simple. The split one is better because it's a pun. Yeah, that's a solid choice. Let me present it with a friendly tone and maybe add an emoji to keep it light. I should also ask if she wants another one to keep the conversation going.
</think>

Why did the banana go to the doctor? 🍌  
Because it had a *split* personality! 😄  

Want another one? I’ve got more where that came from!
CURRENT STATE:  [HumanMessage(content="Hi, I'm Ashley", additional_kwargs={}, response_metadata={}), AIMessage(content='<think>\nOkay, the user introduced themselv

Enter:  exit


Conversation saved to logging.txt
